# Tutorial: Palette catalog summaries

## Audience
- Python learners who want to summarize a small CSV catalog with only the standard library.

## Prerequisites
- Basic Python: functions, dictionaries, sets and lists.
- A UTF-8 CSV catalog with `Primary (Hex)` and `Product Type` columns (here `/inputs/colors.csv`).

## Learning goals
- Read a CSV by header names with `csv.DictReader`.
- Normalize text values (strip whitespace, uppercase hex colors).
- Group rows by a key and sort the groups deterministically.

## Outline

1. Setup
2. Normalize one row
3. Group the catalog with `summarize_palettes`
4. Summarize the real catalog
5. Exercises

In [ ]:
# Setup: standard library only
from __future__ import annotations

import csv
from collections import defaultdict
from io import StringIO

CATALOG_PATH = '/inputs/colors.csv'

## Step 1 - Normalize one row

Colors may have stray spaces or lowercase letters, so `' #2563eb '` and `'#2563EB'` should land in the same group. Product types are only stripped.

In [ ]:
def normalize_row(row: dict[str, str]) -> tuple[str, str]:
    primary = (row.get('Primary (Hex)') or '').strip().upper()
    product_type = (row.get('Product Type') or '').strip()
    return primary, product_type

normalize_row({'Primary (Hex)': ' #2563eb ', 'Product Type': ' Micro SaaS '})

## Step 2 - Group the catalog

`summarize_palettes` reads the CSV by its headers, counts every row (duplicates included) per normalized primary color, collects the sorted unique product types, and orders groups by count descending, then primary ascending. A header-only file gives `[]`.

In [ ]:
def summarize_palettes(csv_path: str) -> list[dict[str, object]]:
    counts: dict[str, int] = defaultdict(int)
    product_types: dict[str, set[str]] = defaultdict(set)
    with open(csv_path, encoding='utf-8', newline='') as handle:
        for row in csv.DictReader(handle):
            primary, product_type = normalize_row(row)
            counts[primary] += 1
            product_types[primary].add(product_type)
    groups = [
        {
            'primary': primary,
            'count': counts[primary],
            'product_types': sorted(product_types[primary]),
        }
        for primary in counts
    ]
    groups.sort(key=lambda group: (-group['count'], group['primary']))
    return groups

## Step 3 - Summarize the real catalog

Show only the top groups to keep output short.

In [ ]:
palette_summary = summarize_palettes(CATALOG_PATH)
len(palette_summary), [(g['primary'], g['count']) for g in palette_summary[:5]]

## Exercises

- Predict how many groups a header-only CSV produces, then check it.
- Add a duplicate row and a lowercase color; confirm the counts change as expected.
- Common mistake: splitting lines on commas breaks quoted names like `"Shop, Retail"`; `csv.DictReader` handles them.

In [ ]:
# Exercise check: one row in mixed order, case and quoting
import os
import tempfile

sample = 'Product Type,Primary (Hex)\n"Shop, Retail", #abcdef \nBlog,#ABCDEF\nBlog,#ABCDEF\n'
with tempfile.TemporaryDirectory() as tmp_dir:
    sample_path = os.path.join(tmp_dir, 'sample.csv')
    with open(sample_path, 'w', encoding='utf-8', newline='') as handle:
        handle.write(sample)
    sample_summary = summarize_palettes(sample_path)
sample_summary